# Multi-day pipeline → indicator table

Reads raw per-day folders from `data/flight_data/` and produces the
**model-ready indicator table** consumed by `5g_foraging_effect_model.ipynb`,
`exposure_analysis.ipynb`, `indicator_validation.ipynb`, etc.

## Pipeline flow

```
data/flight_data/<date>_system_<sys>/      <- raw PATS-C output (source of truth)
    detections.csv, flight_tracks.csv
                         │
                         ▼
       multi_day_pipeline.ipynb            <- THIS NOTEBOOK
                         │
        ┌────────────────┼─────────────────┐
        ▼                ▼                 ▼
  daily_summary    per_track_           indicators_daily.csv
   (n_v3,           indicators           ALL 6 indicators
    re_ratio_v3,    (per-track v3,       per (date, system)
    n_returns,      tortuosity,          + dbm + data transfer
    n_v1, n_v2)     hive_return)         + condition labels
                         │
                         ▼
     flower_visit_pipeline.ipynb          <- run separately (slow)
                         │
                         ▼
       flower_visit_summary.csv
       (mean_handling_time_s,
        n_distinct_flowers)
                         │
        ┌────────────────┴────────────────┐
        ▼                                 ▼
exposure_analysis.ipynb           5g_foraging_effect_model.ipynb
   (figures)                          (pre-registered verdict)
        │                                 │
        └──────────┬──────────────────────┘
                   ▼
          paper/report
```

## STATUS

### WORKS
- Raw data ingestion from `data/flight_data/<date>_system_<sys>/`
- v1 / v2 / v3 hive-exit classification (matches `components.ipynb`)
- Per-track `tortuosity` and `hive_return` flags
- Daily aggregates: n_tracks, n_v3, n_returns, re_ratio_v3, median trip duration
- **NEW**: per-(date, system) path_tortuosity and ifi_cv
- **NEW**: join with `flower_visit_summary.csv` for mean_handling_time_s and n_distinct_flowers
- **NEW**: load dBm + data transfer files, aggregate per (date, system)
- **NEW**: final `indicators_daily.csv` with all 6 sign-aligned indicators ready for the model

### PENDING (you decide)
- Verify sensor → system mapping for dBm (currently `sensor 6 → sys 900`, `sensor 4 → sys 939` based on signal strength gradient)
- Add indoor T / RH / light from greenhouse sensors if available (currently only outdoor KNMI weather is wired in)
- Audio data is scrapped per scope decision

## How to run

1. Make sure `data/flight_data/` is up to date (raw PATS-C folders).
2. Run `flower_visit_pipeline.ipynb` to refresh `flower_visit_summary.csv`
   (only needed when the raw data changes).
3. Run this notebook top-to-bottom. It produces:
   - `data/multi_day/daily_summary.csv`
   - `data/multi_day/per_track_indicators.csv`
   - `data/multi_day/indicators_daily.csv` ← the file the model consumes

Anything that downstream notebooks load from `data/multi_day/` is built here.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


## 1. Configuration

`DATES` filters which dates to process. Set to `None` to include every cached date.

`HIVE_BY_SYSTEM` is the calibrated hive position per camera (from `calibration_and_validation.ipynb`, section 3). Update if your camera moves.


In [2]:
# --- Date filter ---------------------------------------------------------
DATES = None  # None = all available dates; or e.g. pd.date_range("2026-04-19","2026-05-09")
SYSTEM_IDS = [900, 939]

# --- Hive positions per camera (from hive-calibration cell) -------------
HIVE_BY_SYSTEM = {
    900: np.array([-0.04,  -0.665, -1.195]),
    939: np.array([-0.086, -0.828, -1.045]),
}

# --- Classifier thresholds (must match components.ipynb) ----------------
TOL          = 0.10   # m, return + v1-exit tolerance
TAIL_FRAMES  = 10     # frames at end of track to inspect for return
HEAD_FRAMES  = 5      # frames at start to inspect for exit / fit velocity
EXIT_TOL_V2  = 0.20   # m, v2 closest-approach tolerance
MIN_SPEED    = 0.5    # m/s, v2 minimum initial speed
MAX_LAG_S    = 2.0    # s, v2 maximum back-extrapolation lag
FPS          = 60.0   # PATS-C frame rate

# --- Paths ---------------------------------------------------------------
# Read raw recordings straight from data/flight_data/  (cache is bypassed)
DATA_BASE = Path("../../../data/flight_data")
OUT_DIR   = Path("data/multi_day")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source: {DATA_BASE.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")


Cache : /Users/jaspe/Projects/Claude/Bumblebee-monitoring/src/flight_analysis/pats/cache
Output: /Users/jaspe/Projects/Claude/Bumblebee-monitoring/src/flight_analysis/pats/data/multi_day


## 2. Classifier helpers

Self-contained versions of the classifiers and indicators. Same logic as `components.ipynb`, just packaged into pure functions so the loop below can call them per track.


In [3]:
STRONG_NO = {"closest_in_future", "too_far", "too_much_lag"}
WEAK_NO   = {"slow", "short_track"}


def valid_xyz(trk):
    """(N, 3) array of valid positions, or None if there aren't any."""
    if "pos_valid_insect" in trk.columns:
        trk = trk[trk["pos_valid_insect"] == 1]
    if len(trk) == 0:
        return None
    return trk[["posX_insect", "posY_insect", "posZ_insect"]].to_numpy(dtype=float)


def is_hive_return(xyz, hive, tol=TOL, tail=TAIL_FRAMES):
    if xyz is None or len(xyz) == 0:
        return False, np.nan
    tail_xyz = xyz[-tail:]
    d = float(np.min(np.linalg.norm(tail_xyz - hive, axis=1)))
    return (d <= tol), d


def is_hive_exit_v1(xyz, hive, tol=TOL, head=HEAD_FRAMES):
    if xyz is None or len(xyz) == 0:
        return False, np.nan
    head_xyz = xyz[:head]
    d = float(np.min(np.linalg.norm(head_xyz - hive, axis=1)))
    return (d <= tol), d


def is_hive_exit_v2(xyz, hive, head_n=HEAD_FRAMES, exit_tol=EXIT_TOL_V2,
                    min_speed=MIN_SPEED, max_lag_s=MAX_LAG_S, fps=FPS):
    if xyz is None or len(xyz) < head_n:
        return False, "short_track"
    head = xyz[:head_n]
    p0   = head[0]
    t = np.arange(head_n) / fps
    v = np.array([np.polyfit(t, head[:, k], 1)[0] for k in range(3)])
    speed = float(np.linalg.norm(v))
    if speed < min_speed:
        return False, "slow"
    rel    = p0 - hive
    t_star = -float(np.dot(v, rel)) / (speed ** 2)
    if t_star > 0:
        return False, "closest_in_future"
    if abs(t_star) > max_lag_s:
        return False, "too_much_lag"
    d_min = float(np.linalg.norm(p0 + v * t_star - hive))
    return (d_min <= exit_tol), ("ok" if d_min <= exit_tol else "too_far")


def path_tortuosity(xyz):
    """arc length / end-to-end displacement. >= 1.0; 1.0 means perfect line."""
    if xyz is None or len(xyz) < 2:
        return np.nan
    arc = float(np.linalg.norm(np.diff(xyz, axis=0), axis=1).sum())
    end_to_end = float(np.linalg.norm(xyz[-1] - xyz[0]))
    return np.nan if end_to_end < 1e-6 else arc / end_to_end


## 3. Process every cached `(date, system_id)` pair

For each cached pair, load detections + per-uid tracks, classify each track, and accumulate two tables:
- per-track rows (one row per flight)
- per-day rows (one row per date+system aggregate)

Trip duration is computed greedily within each day+camera: each v3 exit is matched to the next un-matched return after it.


In [4]:
pat = re.compile(r"^(\d{4}-\d{2}-\d{2})_system_(\d+)$")

target_dates = None
if DATES is not None:
    target_dates = {pd.Timestamp(d).date().isoformat() for d in DATES}

per_track_rows = []
daily_rows     = []

day_sys_folders = sorted(DATA_BASE.glob("*_system_*"))
print(f"Found {len(day_sys_folders)} day-system folders in {DATA_BASE}")

for d in day_sys_folders:
    m = pat.match(d.name)
    if not m:
        continue
    date_str, sys_str = m.groups()
    sys_id = int(sys_str)
    if sys_id not in SYSTEM_IDS:
        continue
    if target_dates is not None and date_str not in target_dates:
        continue
    if sys_id not in HIVE_BY_SYSTEM:
        print(f"  skipping {d.name} - no hive position for system {sys_id}")
        continue

    det_csv = d / "detections.csv"
    ft_csv  = d / "flight_tracks.csv"
    if not det_csv.exists() or not ft_csv.exists():
        print(f"  skipping {d.name} - missing detections.csv or flight_tracks.csv")
        continue

    dets = pd.read_csv(det_csv)
    dets["ts"] = pd.to_datetime(dets["datetime"],
                                format="%Y%m%d_%H%M%S", errors="coerce")

    # Load all tracks once and group by uid - much faster than reading
    # one CSV per track from cache/<date>_system_<sys>/tracks/.
    ft = pd.read_csv(ft_csv, usecols=["detection_uid", "pos_valid_insect",
                                       "posX_insect", "posY_insect", "posZ_insect",
                                       "elapsed"])
    ft_by_uid = dict(tuple(ft.groupby("detection_uid")))

    hive = HIVE_BY_SYSTEM[sys_id]
    n_tracks = n_ret = n_v1 = n_v2 = n_v3 = 0
    exit_ts  = []
    return_ts = []

    for _, det in dets.iterrows():
        uid = int(det["uid"])
        trk = ft_by_uid.get(uid)
        if trk is None or len(trk) == 0:
            continue
        # ensure frame order
        trk = trk.sort_values("elapsed")
        xyz = valid_xyz(trk)
        if xyz is None or len(xyz) == 0:
            continue
        n_tracks += 1

        is_ret, _   = is_hive_return(xyz, hive)
        is_v1, _    = is_hive_exit_v1(xyz, hive)
        is_v2, rsn  = is_hive_exit_v2(xyz, hive)
        is_v3       = is_v2 or (is_v1 and rsn in WEAK_NO)
        tort        = path_tortuosity(xyz)

        if is_ret:
            n_ret += 1
            return_ts.append(det["ts"])
        if is_v1: n_v1 += 1
        if is_v2: n_v2 += 1
        if is_v3:
            n_v3 += 1
            exit_ts.append(det["ts"])

        per_track_rows.append({
            "date":          date_str,
            "system_id":     sys_id,
            "uid":           uid,
            "ts":            det["ts"],
            "n_samples":     len(xyz),
            "hive_return":   bool(is_ret),
            "hive_exit_v1":  bool(is_v1),
            "hive_exit_v2":  bool(is_v2),
            "hive_exit_v3":  bool(is_v3),
            "v2_reason":     rsn,
            "tortuosity":    tort,
        })

    # --- Trip duration (greedy match v3 exit -> next un-matched return) -
    ex_sorted = sorted(t for t in exit_ts   if pd.notna(t))
    rt_sorted = sorted(t for t in return_ts if pd.notna(t))
    used  = [False] * len(rt_sorted)
    trips = []
    for et in ex_sorted:
        for j, rt in enumerate(rt_sorted):
            if used[j] or rt <= et:
                continue
            trips.append((rt - et).total_seconds())
            used[j] = True
            break

    median_trip = float(np.median(trips)) if trips else np.nan
    mean_trip   = float(np.mean(trips))   if trips else np.nan

    daily_rows.append({
        "date":          date_str,
        "system_id":     sys_id,
        "n_tracks":      n_tracks,
        "n_returns":     n_ret,
        "n_v1":          n_v1,
        "n_v2":          n_v2,
        "n_v3":          n_v3,
        "re_ratio_v3":   (n_ret / n_v3) if n_v3 else np.nan,
        "median_trip_s": median_trip,
        "mean_trip_s":   mean_trip,
        "n_matched":     len(trips),
    })

    print(f"  {date_str}  sys {sys_id:3d}: "
          f"tracks={n_tracks:5d}  v3={n_v3:4d}  returns={n_ret:4d}  "
          f"trips={len(trips):4d}")

daily_summary = (pd.DataFrame(daily_rows)
                 .sort_values(["date", "system_id"])
                 .reset_index(drop=True))
per_track     = pd.DataFrame(per_track_rows)

print("\n=== summary ===")
print(daily_summary.to_string(index=False))


Found 43 cached folders
  2026-04-13  sys 900: tracks=  369  v3=  26  returns=  51  trips=  26
  2026-04-19  sys 900: tracks= 1080  v3= 152  returns= 226  trips= 152
  2026-04-20  sys 900: tracks= 1283  v3= 182  returns= 253  trips= 181
  2026-04-20  sys 939: tracks=  280  v3=   9  returns=   1  trips=   1
  2026-04-21  sys 900: tracks=   21  v3=   1  returns=   0  trips=   0
  2026-04-21  sys 939: tracks=    0  v3=   0  returns=   0  trips=   0
  2026-04-22  sys 900: tracks=  846  v3= 110  returns= 139  trips= 110
  2026-04-22  sys 939: tracks=  659  v3=  87  returns=  91  trips=  86
  2026-04-23  sys 900: tracks=  793  v3= 109  returns= 146  trips= 109
  2026-04-23  sys 939: tracks=  643  v3=  90  returns=  82  trips=  79
  2026-04-24  sys 900: tracks=  898  v3= 111  returns= 164  trips= 111
  2026-04-24  sys 939: tracks=  534  v3=  47  returns=  37  trips=  36
  2026-04-25  sys 900: tracks=  842  v3= 105  returns= 183  trips= 105
  2026-04-25  sys 939: tracks=  422  v3=  27  returns

## 4. Save outputs

Two CSVs in `data/multi_day/`. `exposure_analysis.ipynb` reads from here.


In [5]:
daily_summary.to_csv(OUT_DIR / "daily_summary.csv", index=False)
per_track.to_csv(OUT_DIR / "per_track_indicators.csv", index=False)

print(f"Wrote:")
print(f"  {OUT_DIR / 'daily_summary.csv'}        ({daily_summary.shape[0]} rows)")
print(f"  {OUT_DIR / 'per_track_indicators.csv'} ({per_track.shape[0]} rows)")


Wrote:
  data/multi_day/daily_summary.csv        (42 rows)
  data/multi_day/per_track_indicators.csv (33661 rows)


## 4. Per-(date, system) `path_tortuosity` and `ifi_cv`

`path_tortuosity` — daily *median* of the per-track `tortuosity` column
(across all tracks, not just v3 exits). Rough paths → higher value.

`ifi_cv` — coefficient of variation of the inter-foraging interval
between consecutive v3 hive exits, computed within each (date, system).
Erratic timing → higher CV.


In [ ]:
# path_tortuosity per (date, system) - daily median of tortuosity across all tracks
tort = (per_track.groupby(["date", "system_id"])["tortuosity"]
                  .median()
                  .reset_index(name="path_tortuosity"))

# ifi_cv per (date, system) - CV of inter-exit interval among v3 exits
def _ifi_cv(group):
    ts = group.loc[group["hive_exit_v3"], "ts"].sort_values()
    if len(ts) < 3: return np.nan
    iv = ts.diff().dropna().dt.total_seconds()
    if iv.mean() == 0: return np.nan
    return float(iv.std() / iv.mean())

ifi = (per_track.groupby(["date", "system_id"])
                .apply(_ifi_cv, include_groups=False)
                .reset_index(name="ifi_cv"))

# Merge onto daily_summary
indicators = daily_summary.merge(tort, on=["date","system_id"], how="left")
indicators = indicators.merge(ifi, on=["date","system_id"], how="left")
print(f"After tortuosity + ifi_cv merge: {indicators.shape}")
print(indicators.head(3).to_string())


## 5. Join flower-visit indicators

`mean_handling_time_s` and `n_distinct_flowers` come from
`flower_visit_pipeline.ipynb`. If that notebook hasn't been run yet,
this cell warns and the columns are filled with NaN.


In [ ]:
fv_path = OUT_DIR / "flower_visit_summary.csv"
if fv_path.exists():
    fv = pd.read_csv(fv_path)
    # flower_visit_summary stores date as string; align dtypes
    fv["date"] = pd.to_datetime(fv["date"]).dt.strftime("%Y-%m-%d")
    indicators["date"] = pd.to_datetime(indicators["date"]).dt.strftime("%Y-%m-%d")
    indicators = indicators.merge(
        fv[["date", "system_id", "mean_handling_time_s", "n_distinct_flowers",
            "median_handling_time_s", "n_visits", "revisit_rate"]],
        on=["date", "system_id"], how="left")
    print(f"Joined flower_visit_summary ({len(fv)} rows). Indicator table now {indicators.shape}")
else:
    print(f"!!! {fv_path} not found.  Run flower_visit_pipeline.ipynb first.")
    indicators["mean_handling_time_s"] = np.nan
    indicators["n_distinct_flowers"]   = np.nan
print(indicators.head(3).to_string())


## 6. dBm + data transfer

Both files live in `../../../data/`. We sample at 10-minute intervals;
aggregate to per (date, system) — mean and max dBm per system per day,
plus mean data-transfer rate per day (one value across both systems
because the transmitter feed is shared).

### Sensor → system mapping (PROVISIONAL)
- `sensor 6 → system 900` (closer to transmitter, higher mean dBm in raw data)
- `sensor 4 → system 939` (farther, lower mean dBm)

Confirm and change `SENSOR_TO_SYSTEM` below if you know better.


In [ ]:
import glob

DATA_BASE = Path("../../../data")
SENSOR_TO_SYSTEM = {6: 900, 4: 939}   # PROVISIONAL — confirm

# locate latest dBm file (glob pattern in case the timestamp changes)
dbm_candidates = sorted(DATA_BASE.glob("Power levels (dBm)*.csv"))
xfer_candidates = sorted(DATA_BASE.glob("Data transfer*.csv"))
print(f"dBm files found: {[p.name for p in dbm_candidates]}")
print(f"data-transfer files found: {[p.name for p in xfer_candidates]}")

dbm = pd.read_csv(dbm_candidates[-1])
dbm["timestamp"] = pd.to_datetime(dbm["timestamp"])
dbm["date"]      = dbm["timestamp"].dt.strftime("%Y-%m-%d")
dbm["system_id"] = dbm["sensor"].map(SENSOR_TO_SYSTEM)

dbm_daily = (dbm.dropna(subset=["system_id"])
                .groupby(["date", "system_id"])
                .agg(mean_dbm=("dBm","mean"),
                     max_dbm =("dBm","max"),
                     median_dbm=("dBm","median"))
                .reset_index())
dbm_daily["system_id"] = dbm_daily["system_id"].astype(int)
print(f"dBm aggregated: {dbm_daily.shape}")
print(dbm_daily.head().to_string())

xfer = pd.read_csv(xfer_candidates[-1])
xfer["timestamp"] = pd.to_datetime(xfer["timestamp"])
xfer["date"]      = xfer["timestamp"].dt.strftime("%Y-%m-%d")
# "data" column looks like "270 Mb/s" - strip units
xfer["mbps"] = (xfer["data"].astype(str).str.extract(r"([\d.]+)")[0].astype(float))
xfer_daily = (xfer.groupby("date")
                  .agg(mean_mbps=("mbps","mean"),
                       max_mbps =("mbps","max"))
                  .reset_index())
print(f"\ndata-transfer aggregated: {xfer_daily.shape}")
print(xfer_daily.head().to_string())

indicators = indicators.merge(dbm_daily, on=["date","system_id"], how="left")
indicators = indicators.merge(xfer_daily, on=["date"], how="left")
print(f"\nWith dBm + transfer: {indicators.shape}")


## 7. Sign-align indicators (positive = 5G impairs) and add condition labels

Build the final `indicators_daily.csv`. Six indicators, all sign-aligned
so positive values mean "consistent with the H1 hypothesis that 5G impairs
foraging". Then add the `condition` column from `CYCLE_ANCHOR = 2026-04-23`.


In [ ]:
CYCLE_ANCHOR = pd.Timestamp("2026-04-23")
def label_date(d):
    d = pd.Timestamp(d)
    if d < CYCLE_ANCHOR: return "BASELINE"
    return "ON" if ((d - CYCLE_ANCHOR).days // 3) % 2 == 0 else "OFF"

indicators["date_ts"]   = pd.to_datetime(indicators["date"])
indicators["condition"] = indicators["date_ts"].apply(label_date)
indicators["cycle"]     = ((indicators["date_ts"] - CYCLE_ANCHOR).dt.days // 6).clip(lower=-1)

# Sign-aligned indicators (positive = impairment direction)
indicators["neg_exit_count"] = -indicators["n_v3"].astype(float)
indicators["neg_re_ratio"]   = -indicators["re_ratio_v3"]
# path_tortuosity, ifi_cv, mean_handling_time_s, n_distinct_flowers retained as-is
# (higher = more impairment for first 3; n_distinct_flowers we flip below)

INDICATORS = [
    "neg_exit_count",
    "neg_re_ratio",
    "path_tortuosity",
    "ifi_cv",
    "mean_handling_time_s",
    "n_distinct_flowers",   # NOTE: keeping raw direction; the model decides sign per fit
]

# Columns to keep
out_cols = (["date", "system_id", "condition", "cycle"]
            + INDICATORS
            + ["n_v3", "n_returns", "re_ratio_v3", "median_trip_s", "mean_trip_s",
               "n_visits", "mean_dbm", "max_dbm", "median_dbm",
               "mean_mbps", "max_mbps"])
out_cols = [c for c in out_cols if c in indicators.columns]
indicators_daily = indicators[out_cols].sort_values(["system_id", "date"]).reset_index(drop=True)

OUT_PATH = OUT_DIR / "indicators_daily.csv"
indicators_daily.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  ({len(indicators_daily)} rows, {len(indicators_daily.columns)} cols)")
print()
print("=== indicators_daily.csv preview ===")
print(indicators_daily.head(6).to_string())
print()
print("Coverage by (system, condition):")
print(indicators_daily.groupby(["system_id", "condition"]).size().to_string())
